# Streaming Billing — Pillar 4: Feature Store + Training Loop

End-to-end notebook tying MongoDB Atlas to the offline ML side of the demo:

1. **Streaming feature pipeline** — PySpark Structured Streaming reads `transactions` from the
   MongoDB change stream via the **MongoDB Spark Connector v10.x**, aggregates per-customer
   rolling features (1h / 24h / 7d / 30d / 90d), and writes them to **Delta Lake** as the
   offline store.
2. **Training set assembly** — joins offline features with labels from
   `quarantine_cases_history` (analyst dispositions) to produce a supervised training matrix.
3. **Model training** — sklearn **IsolationForest** + **MLflow** experiment tracking.
4. **Inference & write-back** — scores every customer's latest feature row and `$set`s the
   `model_score`/`model_version` back onto the online `features` collection so the
   dashboard's freshness tile lights up.
5. **Stream-lag tile** — measures the time delta between a Mongo write and its appearance in
   the Delta sink. This is the same number plotted on the dashboard's *Pipeline lag* tile.

## Runtime

| Component | Version | Notes |
|-----------|---------|-------|
| Python    | 3.11    | matches backend |
| Spark     | 3.5.x   | pyspark[sql] |
| MongoDB Spark Connector | **10.3.0** | `org.mongodb.spark:mongo-spark-connector_2.12` |
| Delta Lake | 3.2.x | `io.delta:delta-spark_2.12:3.2.0` |
| sklearn | 1.5.x | IsolationForest |
| MLflow  | 2.16.x | local file store under `./mlruns` |

Connection inputs come from `backend/.env` (same X.509 credentials as the FastAPI app).

## 0. Configuration

We load the same `.env` the backend uses so this notebook stays in sync with the live cluster.
ADR-007 mandates a single source of truth for connection material.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / "backend" / ".env")

MONGO_URI = os.environ["MONGO_URI"]
MONGO_DB  = os.environ.get("MONGO_DB", "streaming_billing")
X509_PATH = os.environ.get("MONGO_X509_PATH", "")

DELTA_ROOT = Path(os.environ.get("ACME_DELTA_ROOT", REPO_ROOT / "data" / "delta"))
DELTA_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_DELTA   = str(DELTA_ROOT / "features_offline")
LAG_DELTA       = str(DELTA_ROOT / "stream_lag")
CHECKPOINT_ROOT = str(DELTA_ROOT / "_checkpoints")

print("Mongo  :", MONGO_URI.split("@")[-1].split("/")[0])
print("DB     :", MONGO_DB)
print("Delta  :", DELTA_ROOT)

## 1. Spark session

We bring up Spark with the Mongo + Delta JARs resolved at runtime.  In production this would be
an EMR job submitted with `--packages`; here we let Spark resolve from Maven the first time it
boots.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

MONGO_PKG = "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0"
DELTA_PKG = "io.delta:delta-spark_2.12:3.2.0"

builder = (
    SparkSession.builder.appName("acme-billing-features")
        .config("spark.jars.packages", f"{MONGO_PKG},{DELTA_PKG}")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.session.timeZone", "Asia/Kuala_Lumpur")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.version

## 2. Streaming feature pipeline

The connector exposes a Mongo change stream as a structured streaming source
(`format("mongodb")` with `change.stream.publish.full.document.only=true`).

We compute three sliding windows in parallel using `groupBy(window(...))`:

* **24h** rolling spend + count
* **7d**  rolling spend + count + discount-rate
* **30d** quarantine count

Each output goes to its own Delta table partitioned by `customer_bucket = hash % 32` so the
downstream training job can pushdown filter.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, MapType,
)

txn_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id",    StringType(), False),
    StructField("merchant_id",    StringType(), True),
    StructField("amount",         DoubleType(), False),
    StructField("discount_amount", DoubleType(), True),
    StructField("timestamp",      TimestampType(), False),
    StructField("location",       MapType(StringType(), StringType()), True),
])

txn_stream = (
    spark.readStream.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", "transactions")
        .option("change.stream.publish.full.document.only", "true")
        .option("change.stream.lookup.full.document", "updateLookup")
        .schema(txn_schema)
        .load()
        .withColumn("customer_bucket", F.abs(F.hash("customer_id")) % F.lit(32))
)
txn_stream.printSchema()

In [ ]:
feature_24h = (
    txn_stream.withWatermark("timestamp", "30 minutes")
        .groupBy(
            F.window("timestamp", "24 hours", "5 minutes").alias("w"),
            "customer_id", "customer_bucket",
        )
        .agg(
            F.count("*").alias("txn_count_24h"),
            F.sum("amount").alias("spend_24h_myr"),
            F.sum("discount_amount").alias("discount_24h_myr"),
        )
        .select(
            "customer_id", "customer_bucket",
            F.col("w.end").alias("window_end"),
            "txn_count_24h", "spend_24h_myr", "discount_24h_myr",
        )
)

feature_7d = (
    txn_stream.withWatermark("timestamp", "1 hour")
        .groupBy(
            F.window("timestamp", "7 days", "30 minutes").alias("w"),
            "customer_id", "customer_bucket",
        )
        .agg(
            F.count("*").alias("txn_count_7d"),
            F.sum("amount").alias("spend_7d_myr"),
            (F.sum("discount_amount") /
             F.when(F.sum("amount") == 0, F.lit(None)).otherwise(F.sum("amount"))).alias("discount_rate_7d"),
        )
        .select(
            "customer_id", "customer_bucket",
            F.col("w.end").alias("window_end"),
            "txn_count_7d", "spend_7d_myr", "discount_rate_7d",
        )
)

In [ ]:
q_24h = (
    feature_24h.writeStream
        .format("delta")
        .outputMode("append")
        .partitionBy("customer_bucket")
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/feature_24h")
        .start(f"{FEATURE_DELTA}/window_24h")
)

q_7d = (
    feature_7d.writeStream
        .format("delta")
        .outputMode("append")
        .partitionBy("customer_bucket")
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/feature_7d")
        .start(f"{FEATURE_DELTA}/window_7d")
)

print("streams running:", [q.id for q in (q_24h, q_7d)])

### 2.1 Materialize windows back into MongoDB `features` (streaming)

The two streaming queries above sink the 24h / 7d aggregates to Delta
only. The drift detector, the iforest scorer, and the dashboard read
those fields from MongoDB — so we close the loop with **two more
streaming queries off the same source DataFrames** that write upserts
into `features.{customer_id}` via the **Mongo Spark Connector v10.3
streaming sink**.

This is the canonical Spark-native shape for the writeback:

- One streaming source (`txn_stream` from §2), two windowed
  DataFrames (`feature_24h`, `feature_7d`), and **four sinks** total —
  Delta (append-only history) + MongoDB (latest-per-customer upsert)
  for each window.
- The Mongo sink uses `operationType = "update"` with
  `idFieldList = "customer_id"` and `upsertDocument = false`. Spark
  micro-batches translate to PyMongo `$set` on dotted paths; nested
  structs (`quality`, `lineage`) are flattened to dotted-path updates
  by the connector, so we *don't* clobber sibling fields like
  `quality.missing_inputs` set by the FE worker.
- A small `foreachBatch` shim dedupes to the latest `window_end` per
  customer **within each micro-batch**, so out-of-order window
  emissions can't overwrite a fresher value with a stale one.
- Each streaming query carries its own checkpoint
  (`{CHECKPOINT_ROOT}/window_*_to_mongo`) so the writeback recovers
  exactly-once on driver restart.

Provenance: every merged doc is tagged
`quality.computed_via = "spark_batch"`, `quality.confidence = 0.95`,
and `lineage.latest_source_at = window_end` so downstream tooling can
tell Spark-owned values apart from seeded warm-start values
(`computed_via = "seed"`) or FE-worker values (`computed_via =
"online"`).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def _project_24h_for_mongo(df):
    """Shape a feature_24h micro-batch into the canonical Mongo sub-doc.

    Keeps the latest `window_end` per customer in the batch (out-of-order
    micro-batches can't overwrite a fresher value with a stale one), then
    builds the `quality` / `lineage` structs that the connector will
    flatten to dotted-path `$set` updates.
    """
    rn = F.row_number().over(
        Window.partitionBy("customer_id").orderBy(F.col("window_end").desc())
    )
    return (
        df.withColumn("rn", rn)
          .where("rn = 1")
          .drop("rn", "customer_bucket", "discount_24h_myr")  # not in canonical schema
          .withColumn("updated_at", F.current_timestamp())
          .withColumn("quality", F.struct(
              F.lit("spark_batch").alias("computed_via"),
              F.lit(0.95).alias("confidence"),
          ))
          .withColumn("lineage", F.struct(
              F.col("window_end").alias("latest_source_at"),
          ))
          .drop("window_end")
    )


def _project_7d_for_mongo(df):
    """Same as _project_24h_for_mongo, for the 7d window DataFrame."""
    rn = F.row_number().over(
        Window.partitionBy("customer_id").orderBy(F.col("window_end").desc())
    )
    return (
        df.withColumn("rn", rn)
          .where("rn = 1")
          .drop("rn", "customer_bucket")
          .withColumn("updated_at", F.current_timestamp())
          .withColumn("quality", F.struct(
              F.lit("spark_batch").alias("computed_via"),
              F.lit(0.95).alias("confidence"),
          ))
          .withColumn("lineage", F.struct(
              F.col("window_end").alias("latest_source_at"),
          ))
          .drop("window_end")
    )


def _mongo_sink_writer(projector):
    """Return a foreachBatch fn that projects the batch then upserts to Mongo.

    The Mongo Spark Connector v10.3 `format("mongodb")` writer takes the
    nested struct shape and translates it to dotted-path `$set` updates
    keyed on `customer_id`. `upsertDocument=false` keeps insert-ownership
    of new customers with the FE worker / seeder.
    """
    def _write(batch_df, batch_id):
        if batch_df.rdd.isEmpty():
            return
        shaped = projector(batch_df)
        (
            shaped.write.format("mongodb")
              .mode("append")
              .option("connection.uri", MONGO_URI)
              .option("database", MONGO_DB)
              .option("collection", "features")
              .option("operationType", "update")
              .option("idFieldList", "customer_id")
              .option("upsertDocument", "false")
              .save()
        )
    return _write


# --- Streaming sinks: feature_24h → Mongo, feature_7d → Mongo --------------
q_24h_mongo = (
    feature_24h.writeStream
        .foreachBatch(_mongo_sink_writer(_project_24h_for_mongo))
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/feature_24h_mongo")
        .outputMode("update")
        .start()
)

q_7d_mongo = (
    feature_7d.writeStream
        .foreachBatch(_mongo_sink_writer(_project_7d_for_mongo))
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/feature_7d_mongo")
        .outputMode("update")
        .start()
)

print("mongo writeback streams running:", [q.id for q in (q_24h_mongo, q_7d_mongo)])

## 3. Stream-lag sampler

We measure end-to-end lag by stamping each row with `processing_time` (Spark) and comparing
against `timestamp` (event-time on the Mongo write).  This drives the dashboard's
*Pipeline lag p95* tile.  The result lands in its own Delta table that we read in batch.

In [ ]:
lag_stream = (
    txn_stream
        .withColumn("processing_time", F.current_timestamp())
        .withColumn(
            "lag_seconds",
            (F.unix_timestamp("processing_time") - F.unix_timestamp("timestamp")).cast(DoubleType()),
        )
        .select("transaction_id", "customer_id", "timestamp", "processing_time", "lag_seconds")
)

q_lag = (
    lag_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/lag")
        .start(LAG_DELTA)
)
q_lag.id

## 4. Training-set assembly (batch)

For supervised training we need (X, y) where:

* **y** = `1` if the analyst disposition was `true_positive`, `0` if `false_positive`.
  We ignore `escalate` to keep the label clean.
* **X** = hybrid **point-in-time** join:
  * **Delta windows** (`window_24h`, `window_7d`): for each label timestamp we take the
    latest completed Spark window with `window_end <= label_at` (no future leakage for those
    aggregates).
  * **Mongo `features`**: remaining columns (`txn_count_1h`, `discount_rate_30d`, …) still use
    the warm-start rule `features.updated_at <= label_at` (true PIT would need historical
    snapshots or a time-travel feature table).

If Delta directories are missing (streams not run yet), we fall back to the warm-start-only join.


In [ ]:
labels = (
    spark.read.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", "quarantine_cases_history")
        .load()
        .where(F.col("disposition").isin("true_positive", "false_positive"))
        .select(
            "customer_id",
            F.col("resolved_at").alias("label_at"),
            (F.col("disposition") == "true_positive").cast("int").alias("label"),
        )
)
labels.cache()
labels.count()

In [ ]:
from pathlib import Path
from pyspark.sql.window import Window

online_feats = (
    spark.read.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", "features")
        .load()
        .select(
            "customer_id",
            "txn_count_1h", "txn_count_24h", "txn_count_7d",
            "spend_24h_myr", "spend_7d_myr",
            "discount_rate_30d", "quarantine_count_30d",
            "package_value_myr", "spend_to_package_ratio",
            "updated_at",
        )
)

_delta_24 = Path(FEATURE_DELTA) / "window_24h"
_delta_7 = Path(FEATURE_DELTA) / "window_7d"

if _delta_24.is_dir() and _delta_7.is_dir():
    w24 = spark.read.format("delta").load(str(_delta_24))
    w7 = spark.read.format("delta").load(str(_delta_7))
    lb = labels.alias("lb")
    j24 = lb.join(
        w24.alias("w24"),
        (F.col("lb.customer_id") == F.col("w24.customer_id"))
        & (F.col("w24.window_end") <= F.col("lb.label_at")),
        "inner",
    )
    ws24 = Window.partitionBy("lb.customer_id", "lb.label_at").orderBy(F.desc("w24.window_end"))
    pit24 = (
        j24.withColumn("_rn", F.row_number().over(ws24))
        .where(F.col("_rn") == 1)
        .select(
            F.col("lb.customer_id").alias("customer_id"),
            F.col("lb.label_at").alias("label_at"),
            F.col("lb.label").alias("label"),
            F.col("w24.txn_count_24h").alias("pit_txn_count_24h"),
            F.col("w24.spend_24h_myr").alias("pit_spend_24h_myr"),
        )
    )
    j7 = pit24.alias("p").join(
        w7.alias("w7"),
        (F.col("p.customer_id") == F.col("w7.customer_id"))
        & (F.col("w7.window_end") <= F.col("p.label_at")),
        "left",
    )
    ws7 = Window.partitionBy("p.customer_id", "p.label_at").orderBy(F.desc("w7.window_end"))
    pit_both = (
        j7.withColumn("_rn7", F.row_number().over(ws7))
        .where(F.col("_rn7") == 1)
        .select(
            "customer_id",
            "label_at",
            "label",
            "pit_txn_count_24h",
            "pit_spend_24h_myr",
            F.col("w7.txn_count_7d").alias("pit_txn_count_7d"),
            F.col("w7.spend_7d_myr").alias("pit_spend_7d_myr"),
            F.col("w7.discount_rate_7d").alias("pit_discount_rate_7d"),
        )
    )
    training = (
        pit_both.join(online_feats.alias("o"), "customer_id", "inner")
        .where(F.col("o.updated_at") <= F.col("label_at"))
        .select(
            "customer_id",
            "label",
            F.coalesce(F.col("o.txn_count_1h"), F.lit(0)).alias("txn_count_1h"),
            F.coalesce(F.col("pit_txn_count_24h"), F.col("o.txn_count_24h")).alias("txn_count_24h"),
            F.coalesce(F.col("pit_txn_count_7d"), F.col("o.txn_count_7d")).alias("txn_count_7d"),
            F.coalesce(F.col("pit_spend_24h_myr"), F.col("o.spend_24h_myr")).alias("spend_24h_myr"),
            F.coalesce(F.col("pit_spend_7d_myr"), F.col("o.spend_7d_myr")).alias("spend_7d_myr"),
            F.coalesce(F.col("o.discount_rate_30d"), F.lit(0.0)).alias("discount_rate_30d"),
            F.coalesce(F.col("o.quarantine_count_30d"), F.lit(0)).alias("quarantine_count_30d"),
            F.coalesce(F.col("o.package_value_myr"), F.lit(0.0)).alias("package_value_myr"),
            F.coalesce(F.col("o.spend_to_package_ratio"), F.lit(0.0)).alias("spend_to_package_ratio"),
        )
    )
else:
    print("Delta windows missing — warm-start only (online features @ label_at)")
    training = (
        labels.join(online_feats, "customer_id", "inner")
        .where(F.col("updated_at") <= F.col("label_at"))
    )

training.printSchema()
training.count()


## 5. Training — IsolationForest with MLflow tracking

IsolationForest is unsupervised but we evaluate it against the analyst labels as a sanity
check.  The model itself trains on the *full* feature population (no labels) — this matches
the production pattern where most customers are unlabelled.

In [ ]:
import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

FEATURES = [
    "txn_count_1h", "txn_count_24h", "txn_count_7d",
    "spend_24h_myr", "spend_7d_myr",
    "discount_rate_30d", "quarantine_count_30d",
    "package_value_myr", "spend_to_package_ratio",
]

df = training.select("customer_id", "label", *FEATURES).toPandas().fillna(0.0)
X = df[FEATURES].to_numpy()
y = df["label"].to_numpy()

mlflow.set_experiment("streaming_billing_quarantine_iforest")
with mlflow.start_run() as run:
    model = IsolationForest(
        n_estimators=200, contamination="auto",
        max_samples="auto", random_state=42, n_jobs=-1,
    ).fit(X)

    raw = -model.score_samples(X)
    auc = roc_auc_score(y, raw) if len(np.unique(y)) > 1 else float("nan")
    ap  = average_precision_score(y, raw) if len(np.unique(y)) > 1 else float("nan")

    mlflow.log_params({"n_estimators": 200, "random_state": 42})
    mlflow.log_metrics({"roc_auc": auc, "avg_precision": ap, "n_train": len(df)})
    mlflow.sklearn.log_model(model, artifact_path="model", input_example=X[:3])
    run_id = run.info.run_id

MODEL_VERSION = f"iforest_{run_id[:8]}"
print("AUC :", auc, " AP:", ap, " version:", MODEL_VERSION)

## 6. Inference & write-back to MongoDB

**Preferred in production:** run the batch job (same logic as this cell) on a schedule:

`PYTHONPATH=backend python ml/jobs/score_features.py`

For ad-hoc demos you can still execute the cell below: it scores every customer's latest online
feature row and `$set`s:

```
{ model_score: float, model_version: str, scored_at: datetime }
```

The Customer 360 page picks these up immediately. On-demand single-customer scoring is available
from the API: `POST /api/features/score/{customer_id}` (requires backend env
`QUARANTINE_IFOREST_*`).


In [ ]:
from datetime import datetime, timezone
from pymongo import MongoClient, UpdateOne, WriteConcern

client_kwargs = {}
if X509_PATH:
    client_kwargs.update({"tls": True, "tlsCertificateKeyFile": X509_PATH})
client = MongoClient(MONGO_URI, **client_kwargs)
feat_coll = client[MONGO_DB]["features"]

all_feats = pd.DataFrame(list(feat_coll.find({}, {"_id": 0, **{f: 1 for f in FEATURES}, "customer_id": 1}))).fillna(0.0)
scores = -model.score_samples(all_feats[FEATURES].to_numpy())

now = datetime.now(timezone.utc)
ops = [
    UpdateOne(
        {"customer_id": cid},
        {"$set": {"model_score": float(s), "model_version": MODEL_VERSION, "scored_at": now}},
    )
    for cid, s in zip(all_feats["customer_id"], scores)
]
if ops:
    res = feat_coll.with_options(write_concern=WriteConcern("majority")).bulk_write(ops, ordered=False)
    print("matched:", res.matched_count, " modified:", res.modified_count)
client.close()

## 7. Stream-lag snapshot for the dashboard

Reads the recent slice of the lag Delta table, computes p50/p95/p99, and writes one row to
`stream_lag_snapshots`.  The `/api/features/freshness` endpoint surfaces the most recent
snapshot to the dashboard.

In [ ]:
from delta.tables import DeltaTable

lag_df = spark.read.format("delta").load(LAG_DELTA)
if lag_df.head(1):
    cutoff = F.current_timestamp() - F.expr("INTERVAL 15 MINUTES")
    recent = lag_df.where(F.col("processing_time") >= cutoff)
    pctl = recent.selectExpr(
        "percentile(lag_seconds, 0.50) as p50",
        "percentile(lag_seconds, 0.95) as p95",
        "percentile(lag_seconds, 0.99) as p99",
        "count(*) as n",
    ).collect()[0].asDict()
    print(pctl)

    client = MongoClient(MONGO_URI, **client_kwargs)
    client[MONGO_DB]["stream_lag_snapshots"].insert_one({
        "measured_at": datetime.now(timezone.utc),
        "lag_p50": pctl["p50"], "lag_p95": pctl["p95"], "lag_p99": pctl["p99"],
        "sample_size": int(pctl["n"]),
        "model_version": MODEL_VERSION,
    })
    client.close()
else:
    print("lag table empty — let the streams run for a few minutes first")

## 8. Shutdown

Stop the streaming queries gracefully so checkpoints commit cleanly.

In [ ]:
for q in (q_24h, q_7d, q_24h_mongo, q_7d_mongo, q_lag):
    q.stop()
spark.stop()